# Pipeline OCR tiếng Việt (viết ngang, trái sang phải) bằng Pytesseract (detection) + Qwen3-VL-4B-Instruct (recognition)

## 1. Kiểm tra môi trường & GPU

In [1]:
import sys, os
!nvidia-smi
print("\nPython:", sys.version)
print("CWD:", os.getcwd())

Sat Jul 18 18:38:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!sudo apt-get update -qq && sudo apt-get install -y -qq tesseract-ocr tesseract-ocr-vie zstd
!pip install -q pytesseract PyMuPDF Pillow opencv-python-headless matplotlib tqdm requests ollama

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 2.)
debconf: falling back to frontend: Readline
Selecting previously unselected package tesseract-ocr-vie.
(Reading database ... 125186 files and directories currently installed.)
Preparing to unpack .../tesseract-ocr-vie_1%3a4.00~git30-7274cfa-1.1_all.deb ...
Unpacking tesseract-ocr-vie (1:4.00~git30-7274cfa-1.1) ...
Selecting previously unselected package zstd.
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up tesseract-ocr-vie (1:4.00~git30-7274cfa-1.1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (

## 2. Cấu hình đường dẫn & tham số

In [ ]:
from pathlib import Path

WORK_ID    = "HVB_003"
WORK_TITLE = "HVB_003_TITLE"

# Đường dẫn file PDF
PDF_PATH = "/kaggle/input/datasets/james2072/hvb-vie/vie/cong-du-tiep-ky/HVB_003_PDFScan_Viet_Công Dư Tiệp Ký 1+2+3.pdf"
PDF_RENDER_DPI = 300

# Trang bắt đầu (0-indexed) và số trang cần OCR
PAGE_START = 281       # bắt đầu từ trang nào (0-indexed)
PAGE_COUNT = None    # OCR bao nhiêu trang (None = tất cả từ PAGE_START)

# Tesseract config
TESS_LANG = "vie"
TESS_PSM  = 6        # 6 = single block, 3 = auto, 4 = single column
TESS_OEM  = 3        # 3 = LSTM engine

# LLM Corrector (Ollama) — bật/tắt
USE_LLM_CORRECTOR = True
LLM_MODEL_NAME = "qwen3-vl-4b"

# ======================== ĐƯỜNG DẪN ========================
WORK_DIR     = Path("/kaggle/working")
PAGES_DIR    = WORK_DIR / "pages"
VIS_BBOX_DIR = WORK_DIR / "vis_bbox"
OUTPUT_DIR   = WORK_DIR / "output"
OUTPUT_FILE  = OUTPUT_DIR / f"{WORK_ID}_vie_raw.txt"

for d in [PAGES_DIR, VIS_BBOX_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"WORK_ID:    {WORK_ID}")
print(f"WORK_TITLE: {WORK_TITLE}")
print(f"PDF_PATH:   {PDF_PATH}")
print(f"OCR range:  start={PAGE_START}, count={PAGE_COUNT or 'ALL'}")
print(f"Tesseract:  lang={TESS_LANG}, psm={TESS_PSM}, oem={TESS_OEM}")
print(f"LLM:        {'ON' if USE_LLM_CORRECTOR else 'OFF'} ({LLM_MODEL_NAME})")
print(f"OUTPUT:     {OUTPUT_FILE}")


WORK_ID:    HVB_003
WORK_TITLE: HVB_003_TITLE
PDF_PATH:   /kaggle/input/datasets/james2072/hvb-vie/vie/cong-du-tiep-ky/HVB_003_PDFScan_Viet_Công Dư Tiệp Ký 1+2+3.pdf
OCR range:  start=281, count=ALL
Tesseract:  lang=vie, psm=6, oem=3
LLM:        ON (qwen3:4b)
OUTPUT:     /kaggle/working/output/HVB_003_vie_raw.txt


## 3. Cấu hình VLM model local (Ollama & QWEN3-VL-4B)

In [4]:
import subprocess, os

if USE_LLM_CORRECTOR:
    REPO_URL = "https://github.com/tqth/qwen3-vl-4b-hosting.git"
    REPO_DIR = "/kaggle/working/qwen3-vl-4b-hosting"

    if not os.path.exists(REPO_DIR):
        print("Clone repo VLM server ...")
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    else:
        print("Repo đã có sẵn, bỏ qua clone.")

    # Khởi động server
    import time, requests

    log_path = "/kaggle/working/start_server.log"
    proc = subprocess.Popen(
        ["python", "start_server.py", "--model", "qwen3vl"],
        cwd=REPO_DIR,
        stdout=open(log_path, "w"),
        stderr=subprocess.STDOUT,
    )
    print(f"Đã khởi động (PID={proc.pid}), đợi server load model ...")

    for i in range(60):
        try:
            r = requests.get("http://localhost:8000/health", timeout=3)
            if r.status_code == 200:
                print("✓ Server sẵn sàng:", r.json())
                break
        except Exception:
            pass
        time.sleep(15)
        print(f"  ...{(i+1)*15}s, xem log: {log_path}")
    else:
        print("⚠ Timeout, xem log để debug:")
        print(open(log_path).read()[-3000:])
else:
    print("⏭ LLM Corrector TẮT — bỏ qua VLM server")

Clone repo VLM server ...


Cloning into '/kaggle/working/qwen3-vl-4b-hosting'...


Đã khởi động (PID=843), đợi server load model ...
  ...15s, xem log: /kaggle/working/start_server.log
  ...30s, xem log: /kaggle/working/start_server.log
  ...45s, xem log: /kaggle/working/start_server.log
  ...60s, xem log: /kaggle/working/start_server.log
  ...75s, xem log: /kaggle/working/start_server.log
✓ Server sẵn sàng: {'model': 'Qwen/Qwen3-VL-4B-Instruct', 'status': 'ok'}


## 4. Đọc và xử lý PDF thành ảnh (PyMuPDF)

In [ ]:
import fitz  # PyMuPDF
from pathlib import Path

# ===================== INPUT =====================
PDF_PATH = Path("/kaggle/input/datasets/james2072/hvb-vie/vie/cong-du-tiep-ky/HVB_003_PDFScan_Viet_Công Dư Tiệp Ký 1+2+3.pdf")

# ===================== OUTPUT ====================
PAGES_DIR = Path("/kaggle/working/pages")
PAGES_DIR.mkdir(parents=True, exist_ok=True)

# Xóa ảnh cũ nếu có
for f in PAGES_DIR.glob("*.png"):
    f.unlink()

# Lấy tên file làm prefix
WORK_ID = PDF_PATH.stem.split("_PDFScan")[0]   # HVB_005

# ===================== RENDER ====================
doc = fitz.open(PDF_PATH)

DPI = 300
zoom = DPI / 72.0
matrix = fitz.Matrix(zoom, zoom)

print(f"Tổng số trang: {len(doc)}")

for i in range(PAGE_START - 1, len(doc)):
    page = doc[i]

    pix = page.get_pixmap(
        matrix=matrix,
        alpha=False
    )

    output_path = PAGES_DIR / f"{WORK_ID}_p{i+1:04d}.png"

    pix.save(output_path)

    print(f"Đã lưu: {output_path.name}")

doc.close()
print(f"\nHoàn tất! Ảnh được lưu tại: {PAGES_DIR}")

Tổng số trang: 493
Đã lưu: HVB_003_p0281.png
Đã lưu: HVB_003_p0282.png
Đã lưu: HVB_003_p0283.png
Đã lưu: HVB_003_p0284.png
Đã lưu: HVB_003_p0285.png
Đã lưu: HVB_003_p0286.png
Đã lưu: HVB_003_p0287.png
Đã lưu: HVB_003_p0288.png
Đã lưu: HVB_003_p0289.png
Đã lưu: HVB_003_p0290.png
Đã lưu: HVB_003_p0291.png
Đã lưu: HVB_003_p0292.png
Đã lưu: HVB_003_p0293.png
Đã lưu: HVB_003_p0294.png
Đã lưu: HVB_003_p0295.png
Đã lưu: HVB_003_p0296.png
Đã lưu: HVB_003_p0297.png
Đã lưu: HVB_003_p0298.png
Đã lưu: HVB_003_p0299.png
Đã lưu: HVB_003_p0300.png
Đã lưu: HVB_003_p0301.png
Đã lưu: HVB_003_p0302.png
Đã lưu: HVB_003_p0303.png
Đã lưu: HVB_003_p0304.png
Đã lưu: HVB_003_p0305.png
Đã lưu: HVB_003_p0306.png
Đã lưu: HVB_003_p0307.png
Đã lưu: HVB_003_p0308.png
Đã lưu: HVB_003_p0309.png
Đã lưu: HVB_003_p0310.png
Đã lưu: HVB_003_p0311.png
Đã lưu: HVB_003_p0312.png
Đã lưu: HVB_003_p0313.png
Đã lưu: HVB_003_p0314.png
Đã lưu: HVB_003_p0315.png
Đã lưu: HVB_003_p0316.png
Đã lưu: HVB_003_p0317.png
Đã lưu: HVB_003_p03

## 5. Khai triển của các hàm giúp xử lý ký tự gây nhiễu trong ảnh

In [ ]:
def _is_noise_line(s):
    """Kiểm tra dòng có phải rác (số trang, ký tự lạc, nhiễu OCR) không."""
    if not s or len(s) <= 1:
        return True
    if re.fullmatch(r'[\d\s/.,\-–\u2014:]+', s):
        return True
    letters = re.findall(r'[a-zA-Z\u00c0-\u024f\u1e00-\u1eff]', s)
    if len(letters) < 2:
        return True
    words = s.split()
    if len(words) >= 6:
        from collections import Counter
        most_common_count = Counter(words).most_common(1)[0][1]
        if most_common_count / len(words) >= 0.5:
            return True
    return False

def filter_for_alignment(text):
    """
    Lọc sạch văn bản tiếng Việt sau OCR để phục vụ Sentence Alignment:
    - Xóa chỉ số cước chú (phần¹ → phần)
    - Loại bỏ ký tự rác/nhiễu OCR
    - Chuẩn hóa khoảng trắng
    - Loại bỏ dòng trống/rác
    """
    if not text:
        return ""
    lines = text.split("\n")
    inline_citation_regex = re.compile(
        r"[¹²³⁴⁵⁶⁷⁸⁹⁰†‡]+|(?<=[a-zA-Z\u00c0-\u024f\u1e00-\u1eff])[\(\[\{]\d+[\)\]\}]"
    )
    strict_vietnamese_regex = re.compile(
        r"[^a-zA-Z0-9\u00c0-\u024f\u1e00-\u1eff\u0300-\u036f\s\.,:;?!\-–—\(\)\[\]\"'""''/]",
        flags=re.UNICODE
    )
    clean_lines = []
    for line in lines:
        line_str = line.strip()
        if not line_str:
            continue
        line_str = inline_citation_regex.sub("", line_str)
        line_str = strict_vietnamese_regex.sub(" ", line_str)
        line_str = re.sub(r"[\(\[\{]\s*[\)\]\}]", "", line_str)
        line_str = re.sub(r"\s+", " ", line_str).strip()
        if line_str and not _is_noise_line(line_str):
            clean_lines.append(line_str)
    return "\n".join(clean_lines)

print("✓ Loaded: filter_for_alignment")

✓ Loaded: filter_for_alignment


## 6. Pipeline tổng — gửi tất cả các ảnh trong file PDF lên VLM, lưu Text output

In [7]:
import requests, base64, re, io
from tqdm.auto import tqdm

SERVER_URL = "http://localhost:8000"

VIE_OCR_PROMPT = """You are a high-accuracy OCR engine for scanned historical Vietnamese (Quốc ngữ) books.

Your task is to transcribe the page exactly as it appears.

Rules:

1. Transcribe ALL visible Vietnamese text.
2. Preserve the original spelling, punctuation, capitalization, accents, line breaks, and historical orthography.
3. Read the page from top to bottom and left to right.
4. Do NOT modernize or normalize the text.
5. Do NOT correct OCR errors by guessing.
6. If one or more characters cannot be recognized confidently, replace only those characters with '?'.
7. Preserve page numbers, headings, titles, and short isolated lines if they are part of the printed page.
8. Ignore only obvious scanning artifacts such as stains, shadows, page borders, folds, and background noise.
9. Do NOT translate, summarize, explain, or annotate.
10. Do NOT output Markdown, code fences, quotation marks, or any extra text.

Output ONLY the transcription.
"""

def pil_image_to_base64(img_pil):
    max_dim = 1536

    w, h = img_pil.size

    if max(w, h) > max_dim:
        scale = max_dim / max(w, h)
        img_pil = img_pil.resize(
            (int(w * scale), int(h * scale)),
            Image.Resampling.LANCZOS
        )

    buffer = io.BytesIO()
    img_pil.save(buffer, format="PNG")

    return base64.b64encode(buffer.getvalue()).decode()


import time
import requests
from requests.exceptions import ReadTimeout, ConnectionError

def recognize_page_vlm(img_pil, timeout=900, retries=4, retry_wait=10):

    img_b64 = pil_image_to_base64(img_pil)

    payload = {
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": VIE_OCR_PROMPT
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/png;base64,{img_b64}"
                        }
                    }
                ]
            }
        ],
        "temperature": 0.0,
        "max_tokens": 4096
    }

    last_error = None

    for attempt in range(1, retries + 1):
        try:
            print(f"  OCR attempt {attempt}/{retries}")

            r = requests.post(
                f"{SERVER_URL}/v1/chat/completions",
                json=payload,
                timeout=timeout
            )

            r.raise_for_status()

            text = r.json()["choices"][0]["message"]["content"]

            text = re.sub(
                r"^```[a-zA-Z]*\n?(.*?)\n?```$",
                r"\1",
                text,
                flags=re.DOTALL
            ).strip()

            return text

        except (ReadTimeout, ConnectionError) as e:
            last_error = e

            if attempt == retries:
                break

            print(f"  ⚠️ Timeout, sẽ thử lại sau {retry_wait}s...")
            time.sleep(retry_wait)

    raise last_error


# ================= OCR =================
from pathlib import Path
from PIL import Image
PAGES_DIR = Path("/kaggle/working/pages")
pages = sorted(PAGES_DIR.glob("*.png"))
all_texts = []

for i, page_path in enumerate(tqdm(pages, desc=f"VLM OCR Việt {WORK_ID}")):
    page_name = Path(page_path).stem
    print(f"\n--- Trang {page_name} ---")

    img = Image.open(page_path).convert("RGB")

    try:
        raw_text = recognize_page_vlm(img)
        final_text = filter_for_alignment(raw_text)

    except Exception as e:
        print(f"❌ Bỏ qua {page_name}: {e}")
        final_text = f"\n===== OCR FAILED: {page_name} =====\n"

    all_texts.append(final_text)

    if (i + 1) % 10 == 0:
        temp_text = "\n\n".join(t.strip() for t in all_texts if t.strip())
        OUTPUT_FILE.write_text(temp_text, encoding="utf-8")
        print(f"💾 Checkpoint: {i + 1} trang")

# Lưu toàn bộ lần cuối cùng (cho các trang lẻ còn lại chưa rơi vào mốc 10)
if all_texts:
    full_text = "\n\n".join(t.strip() for t in all_texts if t.strip())
    OUTPUT_FILE.write_text(full_text, encoding="utf-8")
    
    total_pages = len(all_texts)
    leftover = total_pages % 10
    
    if leftover > 0:
        print(f"\n💾 [Checkpoint Cuối] Đã lưu bù {leftover} trang lẻ cuối cùng.")
        
    print(f"\n✅ Đã hoàn tất xử lý và lưu tổng cộng {total_pages} trang vào: {OUTPUT_FILE}")
    print(f"Độ dài text: {len(full_text):,} ký tự")

VLM OCR Việt HVB_003:   0%|          | 0/213 [00:00<?, ?it/s]


--- Trang HVB_003_p0281 ---
  OCR attempt 1/4

--- Trang HVB_003_p0282 ---
  OCR attempt 1/4

--- Trang HVB_003_p0283 ---
  OCR attempt 1/4

--- Trang HVB_003_p0284 ---
  OCR attempt 1/4

--- Trang HVB_003_p0285 ---
  OCR attempt 1/4

--- Trang HVB_003_p0286 ---
  OCR attempt 1/4

--- Trang HVB_003_p0287 ---
  OCR attempt 1/4

--- Trang HVB_003_p0288 ---
  OCR attempt 1/4

--- Trang HVB_003_p0289 ---
  OCR attempt 1/4

--- Trang HVB_003_p0290 ---
  OCR attempt 1/4
💾 Checkpoint: 10 trang

--- Trang HVB_003_p0291 ---
  OCR attempt 1/4

--- Trang HVB_003_p0292 ---
  OCR attempt 1/4

--- Trang HVB_003_p0293 ---
  OCR attempt 1/4

--- Trang HVB_003_p0294 ---
  OCR attempt 1/4

--- Trang HVB_003_p0295 ---
  OCR attempt 1/4

--- Trang HVB_003_p0296 ---
  OCR attempt 1/4

--- Trang HVB_003_p0297 ---
  OCR attempt 1/4

--- Trang HVB_003_p0298 ---
  OCR attempt 1/4

--- Trang HVB_003_p0299 ---
  OCR attempt 1/4

--- Trang HVB_003_p0300 ---
  OCR attempt 1/4
💾 Checkpoint: 20 trang

--- Trang HVB